In [ ]:
!pip install requests tqdm elasticsearch tiktoken openai

In [ ]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

# Verifica que funcionó viendo el primer elemento
documents[0]

In [ ]:
from elasticsearch import Elasticsearch

# 1. Conectar al cliente
es_client = Elasticsearch('http://localhost:9200') 

# 2. Definir la configuración del índice (Mappings)
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

# 3. Crear el índice
es_client.indices.create(index=index_name, body=index_settings)

In [8]:
from tqdm.auto import tqdm

for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [10]:
query = "How do execute a command on a Kubernetes pod?"

search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            }
        }
    }
}

response = es_client.search(index=index_name, body=search_query)

# Para ver el score del primer resultado:
print(response['hits']['hits'][0]['_score'])

44.50556


In [11]:
query = "How do copy a file to a Docker container?"

search_query_filtered = {
    "size": 3,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp"
                }
            }
        }
    }
}

response_filtered = es_client.search(index=index_name, body=search_query_filtered)

# La pregunta 4 te pide cuál es la PREGUNTA del 3er resultado (índice 2)
print(response_filtered['hits']['hits'][2]['_source']['question'])

How do I copy files from a different folder into docker container’s working directory?


In [12]:
# 1. Extraemos los resultados de la búsqueda anterior en una lista
context_docs = response_filtered['hits']['hits']

# 2. Construimos el bloque de contexto
context_template = """
Q: {question}
A: {text}
""".strip()

context_list = []
for doc in context_docs:
    source = doc['_source']
    context_list.append(context_template.format(question=source['question'], text=source['text']))

context = "\n\n".join(context_list)

# 3. Creamos el prompt final usando la plantilla de la tarea
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the CONTEXT for answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

prompt = prompt_template.format(question=query, context=context)

# Para responder la Pregunta 5, mira la longitud del prompt:
print(len(prompt))

1431


In [13]:
import tiktoken

# 1. Definimos el encoding para gpt-4o
encoding = tiktoken.encoding_for_model("gpt-4o")

# 2. Convertimos el prompt (la cadena que creamos en el paso anterior) en tokens
tokens = encoding.encode(prompt)

# 3. Contamos cuántos tokens hay en la lista
print(len(tokens))

318


In [15]:
from openai import OpenAI
import os

# Configura tu cliente
# Es recomendable usar os.environ.get('OPENAI_API_KEY') si ya la tienes en tu terminal
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

# Enviamos el prompt al modelo gpt-4o
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}]
)

# Extraemos la respuesta de texto
answer = response.choices[0].message.content

print(answer)

To copy a file from your local machine into a Docker container, you can use the `docker cp` command. The basic syntax is as follows:

```shell
docker cp /path/to/local/file_or_directory container_id:/path/in/container
```
